In [38]:
import pandas as pd
import trectools
import ir_datasets
import glob
from tqdm.notebook import tqdm

In [34]:
topics = ["301", "304", "306", "307", "311", "318", "319", "321", "324", "343", "352", "353", "354", "357", "360", "367", "370", "374", "383", "389", "390", "391", "392", "395", "398", "399", "400", "404", "408", "412", "415", "418", "422", "424", "425", "431", "434", "435", "436", "438", "439", "443", "450", "631", "656", "683", "688", "700"]
print(len(topics))

48


In [35]:
qrels = trectools.TrecQrel()
qrels.qrels_data = (
    pd.DataFrame(
        ir_datasets
        .load("disks45/nocr/trec-robust-2004")
        .qrels_iter()
    )
    .rename(columns={
        "query_id": "query",
        "doc_id": "docid",
        "relevance": "rel"
    })
    .assign(q0="Q0")
    .drop(columns=["iteration"])
)

In [42]:
eval_data = []
for runfile in tqdm(glob.glob("../data/external/robust04-runs/*.gz")):
    run = trectools.TrecRun(runfile)
    eval = trectools.TrecEval(run, qrels)
    eval_data.append(
        eval.get_ndcg(depth=10, per_query=True)
        .reset_index()
        .assign(system=run.get_runid())
        .query("query in @topics")
    )

df = pd.concat(eval_data)

  0%|          | 0/110 [00:00<?, ?it/s]

In [53]:
df.to_parquet("../data/interim/run_evals.parquet")